# 3 · A transient operator: the model proposes the solution

Companion to the tutorial *From Hand-Crafted to LLM-Based Variation Operators in Metaheuristics*. It runs offline: the model is a fixed pool of completions, so every number below comes out the same on your machine.

This is the traced iteration of the tutorial, run cell by cell. The artifact is a
TSP tour: the model proposes the solution itself.

**Transient** is the persistence descriptor. Nothing survives the call except the
solution, so every candidate costs one model call, for as long as the search runs.

In [1]:
import _bootstrap
import viz
import tsp_transient as tsp
from llm import MockLLM, load_pool
from search import build_and_validate

coords = [tsp.COORDS[i] for i in range(len(tsp.COORDS))]
viz.tours(coords, [(list(range(len(coords))), 'the instance', 'five cities')])

'<svg xmlns="http://www.w3.org/2000/svg" width="200" height="226" viewBox="0 0 200 226" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="200" height="226" fill="#ffffff"/><g transform="translate(0,0)"><rect x="4" y="4" width="192" height="214" rx="9" fill="none" stroke="#d8dce3" stroke-width="1"/><text x="100.0" y="20.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">the instance</text><polygon points="26.0,188.0 100.0,188.0 174.0,188.0 174.0,40.0 26.0,40.0" fill="#d9456b" fill-opacity="0.07" stroke="#d9456b" stroke-width="1.8" stroke-linejoin="round"/><circle cx="26.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="26.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">0</text><text x="26.0" y="173.0" font-size="9" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="100.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="100.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">1</text><circle cx="174.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="174.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">2</text><circle cx="174.0" cy="40.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="174.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">3</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><text x="100.0" y="210.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="normal">five cities</text></g></svg>'

## What conditions the operator

The prompt carries the schema, the incumbent and its score. Numbers about the
search state, and nothing else: this is **numeric** conditioning, the base channel
of the lens in notebook 5.

In [2]:
incumbent = [0, 2, 3, 4, 1]
print(tsp.Spec().render(incumbent, tsp.tour_len(incumbent), []))

[context] TSP toy instance; minimize Euclidean closed-tour length.
[conditioning] R = permutation of [0, 1, 2, 3, 4] starting at 0; incumbent [0, 2, 3, 4, 1], score 9.236.
[instruction] Emit one lower-length tour if possible.
[format] Use the CANDIDATE envelope.


## What the model returns, in order

Four completions in the pool. Reading them before the run is worth the minute: it
makes the log below predictable instead of magic.

In [3]:
for k, reply in enumerate(load_pool('tsp_pool.txt')):
    payload = reply.split('payload:')[1].split('END_CANDIDATE')[0].strip()
    try:
        tour = tsp.parse(reply)
        tsp.feasible(tour)
        verdict = f'valid, length {tsp.tour_len(tour):.3f}'
    except ValueError as err:
        verdict = f'refused: {err}'
    print(f'{k}: {payload:18s} {verdict}')

0: [0, 1, 4, 4, 2]    refused: feasibility: city [4] duplicated, city [3] missing
1: [0, 1, 2, 3, 4]    valid, length 8.000
2: [0, 2, 1, 3, 4]    valid, length 9.236
3: [0, 3, 1, 2, 4]    valid, length 10.893


In [4]:
viz.tours(coords, [([0, 2, 3, 4, 1], 'incumbent', 'length 9.236'),
                   ([0, 1, 4, 4, 2], 'first sample', 'refused'),
                   ([0, 1, 2, 3, 4], 'second sample', 'length 8.000')])

'<svg xmlns="http://www.w3.org/2000/svg" width="600" height="226" viewBox="0 0 600 226" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="600" height="226" fill="#ffffff"/><g transform="translate(0,0)"><rect x="4" y="4" width="192" height="214" rx="9" fill="none" stroke="#d8dce3" stroke-width="1"/><text x="100.0" y="20.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">incumbent</text><polygon points="26.0,188.0 174.0,188.0 174.0,40.0 26.0,40.0 100.0,188.0" fill="#d9456b" fill-opacity="0.07" stroke="#d9456b" stroke-width="1.8" stroke-linejoin="round"/><circle cx="26.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="26.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">0</text><text x="26.0" y="173.0" font-size="9" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="174.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="174.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">2</text><circle cx="174.0" cy="40.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="174.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">3</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><circle cx="100.0" cy="188.0" r="10" fill="#ffffff" stroke="#d9456b" stroke-width="1.6"/><text x="100.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">1</text><text x="100.0" y="210.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="normal">length 9.236</text></g><g transform="translate(200,0)"><rect x="4" y="4" width="192" height="214" rx="9" fill="none" stroke="#d8dce3" stroke-width="1"/><text x="100.0" y="20.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">first sample</text><polygon points="26.0,188.0 100.0,188.0 26.0,40.0 26.0,40.0 174.0,188.0" fill="#1f9d78" fill-opacity="0.07" stroke="#1f9d78" stroke-width="1.8" stroke-linejoin="round"/><circle cx="26.0" cy="188.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="26.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">0</text><text x="26.0" y="173.0" font-size="9" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="100.0" cy="188.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="100.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">1</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><circle cx="26.0" cy="40.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="26.0" y="44.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">4</text><circle cx="174.0" cy="188.0" r="10" fill="#ffffff" stroke="#1f9d78" stroke-width="1.6"/><text x="174.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">2</text><text x="100.0" y="210.0" font-size="11" fill="#8a8f9a" text-anchor="middle" font-weight="normal">refused</text></g><g transform="translate(400,0)"><rect x="4" y="4" width="192" height="214" rx="9" fill="none" stroke="#d8dce3" stroke-width="1"/><text x="100.0" y="20.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">second sample</text><polygon points="26.0,188.0 100.0,188.0 174.0,188.0 174.0,40.0 26.0,40.0" fill="#2f6fdb" fill-opacity="0.07" stroke="#2f6fdb" stroke-width="1.8" stroke-linejoin="round"/><circle cx="26.0" cy="188.0" r="10" fill="#ffffff" stroke="#2f6fdb" stroke-width="1.6"/><text x="26.0" y="192.0" font-size="11" fill="#1f2430" text-anchor="middle" font-weight="600">0</text><text x="26.0" y="17

## The run

In [5]:
best, best_score = tsp.main()

incumbent [0, 2, 3, 4, 1]  length 9.236
  step 0: accepted 8.0
  step 1: rejected 9.236
  step 2: rejected 10.893
best [0, 1, 2, 3, 4]  length 8.000


In [6]:
llm = MockLLM(load_pool('tsp_pool.txt'))
_, _, log = build_and_validate(llm, tsp.tour_len,
                               lambda c, sc, cur, scur: sc < scur,
                               incumbent, tsp.Spec(), budget=3, retries=1)
viz.trajectory(tsp.tour_len(incumbent), log, 'The walk, step by step')

'<svg xmlns="http://www.w3.org/2000/svg" width="620" height="214" viewBox="0 0 620 214" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="620" height="214" fill="#ffffff"/><text x="20.0" y="18.0" font-size="13" fill="#1f2430" text-anchor="start" font-weight="600">The walk, step by step</text><line x1="46" y1="172.0" x2="602" y2="172.0" stroke="#d8dce3" stroke-width="1"/><text x="36.0" y="176.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">8</text><line x1="46" y1="106.0" x2="602" y2="106.0" stroke="#d8dce3" stroke-width="1"/><text x="36.0" y="110.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">9.45</text><line x1="46" y1="40.0" x2="602" y2="40.0" stroke="#d8dce3" stroke-width="1"/><text x="36.0" y="44.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">10.9</text><polyline points="46.0,115.6 231.3,172.0 416.7,172.0 602.0,172.0" fill="none" stroke="#1f2430" stroke-width="1.6" stroke-opacity="0.35"/><circle cx="46.0" cy="115.6" r="5" fill="#ffffff" stroke="#1f2430" stroke-width="1.6"/><text x="46.0" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="231.3" cy="172.0" r="5.5" fill="#1f9d78" fill-opacity="0.9"/><text x="231.3" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">step 0</text><circle cx="416.7" cy="115.6" r="5.5" fill="#d9456b" fill-opacity="0.9"/><text x="416.7" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">step 1</text><circle cx="602.0" cy="40.0" r="5.5" fill="#d9456b" fill-opacity="0.9"/><text x="602.0" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">step 2</text><circle cx="46" cy="204" r="4.5" fill="#1f9d78"/><text x="55.0" y="208.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">accepted</text><circle cx="142" cy="204" r="4.5" fill="#d9456b"/><text x="151.0" y="208.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">rejected</text><circle cx="238" cy="204" r="4.5" fill="#d98324"/><text x="247.0" y="208.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">invalid</text><text x="602.0" y="18.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">lower is better</text></svg>'

Step 0 is the interesting one. The log says *accepted*, and the pool says the
first completion was infeasible. Both are true: the repair happened inside the
step, so three steps consumed four completions. A cost table that counts steps
instead of calls will understate the bill by exactly that much.

In [7]:
class CountingLLM:
    """Same model, but it keeps a tally of how often it was called."""

    def __init__(self, inner):
        self.inner, self.calls = inner, 0

    def sample(self, prompt):
        self.calls += 1
        return self.inner.sample(prompt)

llm = CountingLLM(MockLLM(load_pool('tsp_pool.txt')))
build_and_validate(llm, tsp.tour_len, lambda c, sc, cur, scur: sc < scur,
                   incumbent, tsp.Spec(), budget=3, retries=1)
print(f'3 steps, {llm.calls} model calls')

3 steps, 4 model calls


## What transient costs

Nothing is left behind. Ask for another tour tomorrow and the operator pays again,
because the artifact was the answer, not a method for producing answers. That is
the trade the next notebook turns around.

## The same loop in C++

`cpp/tsp_transient.cpp` reads the same completions and prints the same lines. Both
are compared against `expected/tsp_transient.txt`, which makes the port a test: if
they ever disagree, Algorithm 1 leaves something open. Build it with `make -C cpp`.

In [8]:
import os, subprocess

binary = os.path.join('cpp', 'bin', 'tsp_transient')
if os.path.exists(binary):
    cpp = subprocess.run([binary], capture_output=True, text=True).stdout
    frozen = open(os.path.join('expected', 'tsp_transient.txt')).read()
    print(cpp)
    print('identical to the Python run:', cpp == frozen)
else:
    print('not built — run: make -C cpp')

incumbent [0, 2, 3, 4, 1]  length 9.236
  step 0: accepted 8.0
  step 1: rejected 9.236
  step 2: rejected 10.893
best [0, 1, 2, 3, 4]  length 8.000

identical to the Python run: True


## Try it

1. Start from `[0, 1, 2, 3, 4]`, the optimum. Every candidate is now rejected and
   the operator spends three calls to learn nothing. A stopping rule is not a
   luxury.
2. Add a completion of your own to `fixtures/tsp_pool.txt`, separated by a line
   with three dashes. The C++ demo picks it up too, from the same file.
3. Swap `tour_len` for a version that penalises long first legs. The operator does
   not change; only the evaluator does.

---

Next: [4 · The model writes the heuristic](04_bpp_amortized.ipynb)